# WS04: Semantic Interoperability with Knowledge Graphs

## Overview
This workshop demonstrates how to use **Knowledge Graphs** and **ontologies** to create intelligent, self-configuring applications. You will learn:
- Connect to a SPARQL endpoint (GraphDB in this workshop)
- Query structured knowledge about building entities using SPARQL
- Apply reasoning to enhance query effectiveness
- Automatically generate application configurations based on the retrieved knowledge

## Use Case
Building on the intelligent building scenario from previous workshops, we'll now create a **ventilation controller** that automatically configures itself based on available sensors in each room. The controller will intelligently choose the best control strategy based on sensor availability:
- **Priority 1**: Use CO2 sensors for demand-controlled ventilation
- **Priority 2**: Use presence sensors for occupancy-based control
- **Fallback**: Use a simple timetable schedule if no sensors available


## Step 1: Setup and Configuration

First, let's import the necessary packages and configure the connection to the RDF4J server.

In [21]:
import os
import yaml
from pathlib import Path
from rdf4j_client import SparqlApiClient

# Configure the connection to RDF4J SPARQL endpoint
graphdb_url = "https://ebc-demo-graphdb.eonerc.rwth-aachen.de/repositories/InteroperabilityWorkshop"
query_client = SparqlApiClient(endpoint_url=graphdb_url, credential_path="credential.json")
print("✓ Connected to RDF4J SPARQL endpoint successfully!")

SparqlApiClient (requests-based) initialized. Pointing to: https://ebc-demo-graphdb.eonerc.rwth-aachen.de/repositories/InteroperabilityWorkshop
✓ Connected to RDF4J SPARQL endpoint successfully!


## Step 2: Ontology and Semantic Reasoning

Before we can query our knowledge graph effectively, we need to enrich it with semantic rules and ontologies. In this step, we use the **Brick ontology** to perform logical inference. This allows the knowledge graph to automatically derive additional relationships and properties that are not explicitly stated in the original data.

**Action**: Go to GraphDB GUI → Import → Upload RDF Files → Select the Brick ontology file

This enables semantic reasoning, which helps us understand relationships between entities (e.g., which sensors belong to which rooms, which actuators are available for control).


## Step 3: Information Retrieval via SPARQL

In this step, we will execute SPARQL queries to retrieve information from the knowledge graph.
The following graphic illustrates the query workflow:

<img src="images/query_workflow.png" alt="Query Workflow" width="400">

### Query 1: Find Existing Rooms

Our first query discovers all rooms available in the knowledge graph.

In [22]:
query_rooms = """
    PREFIX rec: <https://w3id.org/rec#>
    SELECT DISTINCT ?room
    WHERE {
        ?room a rec:Room
    }
"""
query1_res = query_client.query(query_string=query_rooms)
print(f"Number of rooms found: {len(query1_res)}")
# Select the first room for our example
room_iri = query1_res.bindings[8]["room"]
print(f"Example room IRI: {room_iri}")

Number of rooms found: 10
Example room IRI: http://example.com/HotelRoom/HotelRoom:room_base_8



### Query 2: Find Available Ventilation Devices and Control Points

For our selected room, we now find the ventilation systems and their controllable datapoints (setpoints or commands).

In [23]:
query_ventilation_devices = f"""
    PREFIX rec:   <https://w3id.org/rec#>
    PREFIX brick: <https://brickschema.org/schema/Brick#>
    PREFIX rdf:   <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

    SELECT ?device ?actuation ?actuation_access
    WHERE {{
      <{room_iri}> a rec:Room .

      ?device a brick:Air_System .
      ?actuation brick:isPointOf ?device .

      OPTIONAL {{
        ?actuation a ?actuation_type ;
                   rdf:value ?actuation_access .
        VALUES ?actuation_type {{ brick:Setpoint brick:Command }}
      }}

      ?actuation (brick:isPointOf|brick:hasLocation|brick:isPartOf)* <{room_iri}> .
    }}
"""
query2_res = query_client.query(query_string=query_ventilation_devices)
print(f"Number of ventilation devices found: {len(query2_res)}")
# Extract the device, actuation point, and access point
if len(query2_res) == 1:
    device = query2_res.bindings[0].get("device")
    actuation = query2_res.bindings[0].get("actuation")
    actuation_access = query2_res.bindings[0].get("actuation_access")
    print(f"Device IRI: {device}\nActuation IRI: {actuation}\nAPI Endpoint: {actuation_access}")

Number of ventilation devices found: 1
Device IRI: http://example.com/FreshAirVentilation/FreshAirVentilation:room_base_8
Actuation IRI: http://example.com/airFlowSetpoint_FreshAirVentilation/airFlowSetpoint_FreshAirVentilation:room_base_8
API Endpoint: https://fiware.eonerc.rwth-aachen.de/v2/entities/FreshAirVentilation:room_base_8/attrs/airFlowSetpoint/value



### Query 3: Check for Available Sensors in the Room

The control strategy depends on what sensors are available. We'll check for sensors in priority order:
1. **Priority 1**: CO2 sensors for demand-controlled ventilation
2. **Priority 2**: Presence sensors for occupancy-based control

Let's start by looking for a CO2 sensor:

In [24]:
query_co2_sensor_availability = f"""
    PREFIX rec: <https://w3id.org/rec#>
    PREFIX brick: <https://brickschema.org/schema/Brick#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

    SELECT ?sensor ?sensor_access
    WHERE {{
        <{room_iri}> a rec:Room .
        ?sensor a brick:CO2_Sensor .
        ?sensor rdf:value ?sensor_access .
        ?sensor (brick:isPointOf|brick:hasLocation|brick:isPartOf)* <{room_iri}>.
    }}
"""
query3_res = query_client.query(query_string=query_co2_sensor_availability)
if len(query3_res) > 0:
    sensor = query3_res.bindings[0].get("sensor")
    sensor_access = query3_res.bindings[0].get("sensor_access")
    print(f"CO2 Sensor found")
    print(f"Sensor IRI: {sensor}\nAPI Endpoint: {sensor_access}")
else:
    print("No CO2 sensor found.")

No CO2 sensor found.



If no CO2 sensor is found, try to find a presence sensor.

In [25]:
query_presence_sensor_availability = f"""
    PREFIX rec: <https://w3id.org/rec#>
    PREFIX brick: <https://brickschema.org/schema/Brick#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

    SELECT ?sensor ?sensor_access
    WHERE {{
        <{room_iri}> a rec:Room .
        ?sensor a brick:Occupancy_Count_Sensor .
        ?sensor rdf:value ?sensor_access .
        ?sensor (brick:isPointOf|brick:hasLocation|brick:isPartOf)* <{room_iri}>.
    }}
"""
query3_res_presence = query_client.query(query_string=query_presence_sensor_availability)
if len(query3_res_presence) > 0:
    sensor = query3_res_presence.bindings[0].get("sensor")
    sensor_access = query3_res_presence.bindings[0].get("sensor_access")
    print(f"Presence Sensor found")
    print(f"Sensor IRI: {sensor}\nAPI Endpoint: {sensor_access}")
else:
    print("No Presence sensor found.")

No Presence sensor found.


## Step 4: Apply the same logic to the whole dataset and generate configuration file

The same queries are implemented in the `ControllerConfiguration` class.
After querying the knowledge graph, a configuration file in YAML format will be generated for deploying controller applications.

In [26]:
from controller_config import ControllerConfiguration
import yaml

configurator = ControllerConfiguration(
    api_client=query_client,
)
configuration = configurator.generate_configuration()


--- Starting Configuration Generation ---
Found 10 rooms. Processing...

Processing Room: http://example.com/HotelRoom/HotelRoom:room_base_1
  -> Found Actuator: https://fiware.eonerc.rwth-aachen.de/v2/entities/FreshAirVentilation:room_base_1/attrs/airFlowSetpoint/value
  -> Found CO2 Sensor: https://fiware.eonerc.rwth-aachen.de/v2/entities/CO2Sensor:room_base_1/attrs/co2/value

Processing Room: http://example.com/HotelRoom/HotelRoom:room_base_5
  -> Found Actuator: https://fiware.eonerc.rwth-aachen.de/v2/entities/FreshAirVentilation:room_base_5/attrs/airFlowSetpoint/value
  -> Found CO2 Sensor: https://fiware.eonerc.rwth-aachen.de/v2/entities/CO2Sensor:room_base_5/attrs/co2/value

Processing Room: http://example.com/HotelRoom/HotelRoom:room_base_6
  -> Found Actuator: https://fiware.eonerc.rwth-aachen.de/v2/entities/FreshAirVentilation:room_base_6/attrs/airFlowSetpoint/value
  -> Found CO2 Sensor: https://fiware.eonerc.rwth-aachen.de/v2/entities/CO2Sensor:room_base_6/attrs/co2/value


In [27]:
# View the generated configuration
print(f"\n--- Generation Complete ---")
print(f"Configuration for {len(configuration)} controllers:")
print(yaml.dump(configuration, sort_keys=False))


--- Generation Complete ---
Configuration for 10 controllers:
- controller_function: Ventilation
  controller_mode: co2
  inputs:
    sensor_access: https://fiware.eonerc.rwth-aachen.de/v2/entities/CO2Sensor:room_base_1/attrs/co2/value
  outputs:
    actuation_access: https://fiware.eonerc.rwth-aachen.de/v2/entities/FreshAirVentilation:room_base_1/attrs/airFlowSetpoint/value
- controller_function: Ventilation
  controller_mode: co2
  inputs:
    sensor_access: https://fiware.eonerc.rwth-aachen.de/v2/entities/CO2Sensor:room_base_5/attrs/co2/value
  outputs:
    actuation_access: https://fiware.eonerc.rwth-aachen.de/v2/entities/FreshAirVentilation:room_base_5/attrs/airFlowSetpoint/value
- controller_function: Ventilation
  controller_mode: co2
  inputs:
    sensor_access: https://fiware.eonerc.rwth-aachen.de/v2/entities/CO2Sensor:room_base_6/attrs/co2/value
  outputs:
    actuation_access: https://fiware.eonerc.rwth-aachen.de/v2/entities/FreshAirVentilation:room_base_6/attrs/airFlowSetp